In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# ── Install all required packages ──────────────────────────────────────────────
!pip install -q torch torchaudio transformers datasets peft bitsandbytes jiwer gradio accelerate librosa
!pip install -q --upgrade transformers peft accelerate

print("✅ All dependencies installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 98.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 99.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires n

In [3]:
import os
import torch
import numpy as np
import gradio as gr
import torchaudio
import librosa

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    BitsAndBytesConfig,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from datasets import load_dataset, Audio
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from jiwer import wer, cer
from peft import LoraConfig, PeftModel, get_peft_model

print("✅ All imports successful.")

✅ All imports successful.


In [6]:
# ── Model & Data ───────────────────────────────────────────────────────────────
BASE_MODEL      = "openai/whisper-small"          # swap for whisper-medium etc.
DATASET_NAME    = "gmenon/slt-lyrics-audio"       # HF Hub dataset id
SAVE_PATH       = "./autolyrics-lora-ckpt"        # where adapters are saved

# ── Training hyper-parameters ─────────────────────────────────────────────────
TRAIN_SAMPLES   = 150    # how many training examples to use (increase if GPU allows)
EVAL_SAMPLES    = 12     # how many validation examples
BATCH_SIZE      = 4
GRAD_ACCUM      = 2      # effective batch = BATCH_SIZE * GRAD_ACCUM
LR              = 1e-4
WARMUP_STEPS    = 10
MAX_STEPS       = 100    # quick run; raise to 300-500 for better results
SAVE_EVAL_STEPS = 50
LOG_STEPS       = 10

# ── LoRA hyper-parameters ─────────────────────────────────────────────────────
LORA_RANK       = 32     # r — controls adapter capacity
LORA_ALPHA      = 64     # scaling factor
LORA_DROPOUT    = 0.05
LORA_TARGETS    = ["q_proj", "v_proj"]   # attention projection layers

# ── Hardware ──────────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = (DEVICE == "cuda")

print(f"Device  : {DEVICE}")
print(f"FP16    : {USE_FP16}")
print(f"Model   : {BASE_MODEL}")
print(f"Dataset : {DATASET_NAME}")

Device  : cuda
FP16    : True
Model   : openai/whisper-small
Dataset : gmenon/slt-lyrics-audio


In [7]:
print("\n[1/3] Loading processor...")
processor = WhisperProcessor.from_pretrained(BASE_MODEL, language="english", task="transcribe")

print("[2/3] Loading dataset...")
raw_ds = load_dataset(DATASET_NAME)
raw_ds = raw_ds.cast_column("audio", Audio(sampling_rate=16000))

# ── Feature extraction + tokenisation ─────────────────────────────────────────
def prepare_sample(batch):
    audio = batch["audio"]
    # log-mel spectrogram
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    # reference transcript — dataset may use 'text' or 'lyrics'
    ref = batch.get("text") or batch.get("lyrics") or ""
    batch["labels"] = processor.tokenizer(ref).input_ids
    return batch

print("[3/3] Preprocessing splits (resampling → spectrogram → tokenise)...")
train_raw = raw_ds["train"].select(range(min(TRAIN_SAMPLES, len(raw_ds["train"]))))
test_raw  = (
    raw_ds["test"].select(range(min(EVAL_SAMPLES, len(raw_ds["test"]))))
    if "test" in raw_ds else train_raw
)

train_ds = train_raw.map(prepare_sample, remove_columns=train_raw.column_names)
eval_ds  = test_raw.map(prepare_sample,  remove_columns=test_raw.column_names)

# ── Dynamic-padding collator ───────────────────────────────────────────────────
@dataclass
class PaddingCollator:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Pad input spectrograms
        input_batch = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_batch, return_tensors="pt")

        # Pad labels; replace padding token id with -100 so loss ignores it
        label_batch = [{"input_ids": f["labels"]} for f in features]
        labels_padded = self.processor.tokenizer.pad(label_batch, return_tensors="pt")
        labels = labels_padded["input_ids"].masked_fill(
            labels_padded.attention_mask.ne(1), -100
        )
        # Strip leading BOS token if present
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

collator = PaddingCollator(processor=processor)
print(f"\n✅ Train samples : {len(train_ds)}")
print(f"✅ Eval  samples : {len(eval_ds)}")


[1/3] Loading processor...


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Generating train split:   0%|          | 0/9538 [00:00<?, ? examples/s]

Generating eval split:   0%|          | 0/507 [00:00<?, ? examples/s]

[3/3] Preprocessing splits (resampling → spectrogram → tokenise)...


Map:   0%|          | 0/150 [00:00<?, ? examples/s]


✅ Train samples : 150
✅ Eval  samples : 150


In [8]:
!pip install -q --upgrade torchao  

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.5 MB/s eta 0:00:0000:0100:01


In [9]:
print("\n[1/2] Loading base model...")

# 8-bit quantisation on GPU to save VRAM; skipped on CPU
quant_cfg = BitsAndBytesConfig(load_in_8bit=True) if DEVICE == "cuda" else None

model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_cfg,
    device_map="auto" if DEVICE == "cuda" else None,
)

# Required when using gradient checkpointing with PEFT
model.config.use_cache          = False
model.config.forced_decoder_ids = None
model.config.suppress_tokens    = []
model.enable_input_require_grads()

print("[2/2] Injecting LoRA adapters...")
lora_cfg = LoraConfig(
    r              = LORA_RANK,
    lora_alpha     = LORA_ALPHA,
    target_modules = LORA_TARGETS,
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    task_type      = "SEQ_2_SEQ_LM",
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
print("\n✅ Model ready for fine-tuning.")


[1/2] Loading base model...


model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

[2/2] Injecting LoRA adapters...
trainable params: 3,538,944 || all params: 245,273,856 || trainable%: 1.4429

✅ Model ready for fine-tuning.


In [10]:
# ── Variable fix ───────────────────────────────────────────────────────────
import types
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from datasets import load_dataset, Audio

MODEL_NAME = "openai/whisper-small"
OUTPUT_DIR = "./whisper-lora-autolyrics"
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 5e-5
MAX_STEPS = 100
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
device = DEVICE
BASE_MODEL = MODEL_NAME

# Rebuild processor
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language="english", task="transcribe")

# Rebuild dataset
raw_dataset = load_dataset("gmenon/slt-lyrics-audio")
raw_dataset = raw_dataset.cast_column("audio", Audio(sampling_rate=16000))

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    target_text = batch.get("text", batch.get("lyrics", ""))
    batch["labels"] = processor.tokenizer(target_text).input_ids
    return batch

train_split = raw_dataset["train"].select(range(min(50, len(raw_dataset["train"]))))
test_split = raw_dataset["test"].select(range(min(5, len(raw_dataset["test"])))) if "test" in raw_dataset else train_split
train_dataset = train_split.map(prepare_dataset, remove_columns=train_split.column_names)
eval_dataset  = test_split.map(prepare_dataset,  remove_columns=test_split.column_names)

# Rebuild collator
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [11]:
import types

# ── Patch (must be defined before get_peft_model) ──────────────────────────
def _patched_forward(
    self,
    input_features=None,
    attention_mask=None,
    decoder_input_ids=None,
    decoder_attention_mask=None,
    decoder_inputs_embeds=None,
    labels=None,
    output_attentions=None,
    output_hidden_states=None,
    return_dict=None,
    **kwargs,
):
    kwargs.pop("input_ids", None)
    kwargs.pop("num_items_in_batch", None)
    with self._enable_peft_forward_hooks(**kwargs):
        kwargs = {k: v for k, v in kwargs.items() if k not in self.special_peft_forward_args}
        return self.base_model(
            input_features=input_features,
            attention_mask=attention_mask,
            decoder_input_ids=decoder_input_ids,
            decoder_attention_mask=decoder_attention_mask,
            decoder_inputs_embeds=decoder_inputs_embeds,
            labels=labels,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
            **kwargs,
        )

# ── Your original Cell 6 code below, unchanged ────────────────────────────
print("\n--- Loading Model and Applying LoRA ---")
model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    device_map="auto" if DEVICE == "cuda" else None
)
model.config.use_cache = False
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.enable_input_require_grads()

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
)
model = get_peft_model(model, peft_config)
model.forward = types.MethodType(_patched_forward, model)  # ← apply patch here
model.print_trainable_parameters()

print("\n--- Defining Evaluation Metric ---")
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer_score = 100 * wer(label_str, pred_str)
    return {"wer": wer_score}

print("\n--- Setting Up Trainer ---")
train_dataset = train_dataset.select_columns(["input_features", "labels"])
eval_dataset = eval_dataset.select_columns(["input_features", "labels"])

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=10,
    max_steps=MAX_STEPS,
    gradient_checkpointing=True,
    fp16=True if device == "cuda" else False,
    eval_strategy="steps",
    per_device_eval_batch_size=BATCH_SIZE,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=25,
    eval_steps=5,
    logging_steps=5,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    label_names=["labels"],
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

print("\n--- Starting Training ---")
trainer.train()

print("\n--- Saving the Finetuned Adapter ---")
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Model and processor successfully saved to {OUTPUT_DIR}")


--- Loading Model and Applying LoRA ---


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


trainable params: 884,736 || all params: 242,619,648 || trainable%: 0.3647

--- Defining Evaluation Metric ---

--- Setting Up Trainer ---


2026-06-04 08:20:47.973476: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780561248.159477      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780561248.216189      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780561248.672821      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780561248.672847      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780561248.672850      58 computation_placer.cc:177] computation placer alr


--- Starting Training ---


Step,Training Loss,Validation Loss,Wer
5,21.806114,11.940454,25400
10,21.490176,11.465324,25500
15,20.014984,10.667137,25500
20,16.465729,9.912811,25300
25,17.375287,9.248077,24700
30,16.453731,8.680593,24500
35,15.405453,8.184726,25200
40,12.840779,7.735528,25400
45,13.775911,7.316177,25500
50,12.987067,6.929920,25600


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its para


--- Saving the Finetuned Adapter ---
Model and processor successfully saved to ./whisper-lora-autolyrics


In [12]:
def run_evaluation(base_ckpt: str, adapter_path: str, eval_data, n_samples: int = 5):
    """
    Compare base Whisper vs LoRA fine-tuned model on n_samples examples.
    Prints WER, CER, and relative WER reduction.
    """
    print("\n--- Loading evaluation models ---")
    base_m = WhisperForConditionalGeneration.from_pretrained(base_ckpt).to(DEVICE)
    base_m.eval()

    lora_m = WhisperForConditionalGeneration.from_pretrained(base_ckpt)
    lora_m = PeftModel.from_pretrained(lora_m, adapter_path).to(DEVICE)
    lora_m.eval()

    references, base_preds, lora_preds = [], [], []
    samples = eval_data.select(range(min(n_samples, len(eval_data))))

    print(f"Running inference on {len(samples)} samples...")
    for sample in samples:
        feats = torch.tensor([sample["input_features"]]).to(DEVICE)

        # Decode reference
        ref_ids = [tok for tok in sample["labels"] if tok != -100]
        references.append(processor.tokenizer.decode(ref_ids, skip_special_tokens=True))

        with torch.no_grad():
            # Base model prediction
            base_out = base_m.generate(input_features=feats)
            base_preds.append(processor.tokenizer.decode(base_out[0], skip_special_tokens=True))

            # LoRA fine-tuned prediction
            lora_out = lora_m.generate(input_features=feats)
            lora_preds.append(processor.tokenizer.decode(lora_out[0], skip_special_tokens=True))

    # ── Metrics ───────────────────────────────────────────────────────────────
    b_wer = wer(references, base_preds)
    l_wer = wer(references, lora_preds)
    b_cer = cer(references, base_preds)
    l_cer = cer(references, lora_preds)

    sep = "-" * 55
    print(f"\n{sep}")
    print(f"{'Metric':<20} {'Base Model':>15} {'LoRA Fine-Tuned':>15}")
    print(sep)
    print(f"{'WER':<20} {b_wer*100:>14.2f}% {l_wer*100:>14.2f}%")
    print(f"{'CER':<20} {b_cer*100:>14.2f}% {l_cer*100:>14.2f}%")
    print(sep)

    if b_wer > 0:
        rel_improvement = (b_wer - l_wer) / b_wer * 100
        status = "✅ TARGET MET" if rel_improvement >= 15 else "⚠️  Below 15% target"
        print(f"Relative WER Reduction : {rel_improvement:.2f}%  {status}")

    print(sep)

    # ── Sample-level output ───────────────────────────────────────────────────
    print("\nSample predictions:")
    for i, (ref, bp, lp) in enumerate(zip(references, base_preds, lora_preds)):
        print(f"\n  Sample {i+1}")
        print(f"  Reference  : {ref}")
        print(f"  Base       : {bp}")
        print(f"  LoRA-Tuned : {lp}")

run_evaluation(BASE_MODEL, OUTPUT_DIR, eval_dataset)


--- Loading evaluation models ---


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Running inference on 5 samples...

-------------------------------------------------------
Metric                    Base Model LoRA Fine-Tuned
-------------------------------------------------------
WER                         2800.00%        2000.00%
CER                        12700.00%        9300.00%
-------------------------------------------------------
Relative WER Reduction : 28.57%  ✅ TARGET MET
-------------------------------------------------------

Sample predictions:

  Sample 1
  Reference  : 
  Base       :  What about now?
  LoRA-Tuned :  What about now?

  Sample 2
  Reference  : 
  Base       :  Men ibland var det en drömm för jag här.
  LoRA-Tuned :  very well.

  Sample 3
  Reference  : 
  Base       :  I'll let you yesterday's child to me
  LoRA-Tuned :  But show yesterday's child to me

  Sample 4
  Reference  : 
  Base       :  I'm sorry.
  LoRA-Tuned :  I'm sorry.

  Sample 5
  Reference  : 
  Base       :  Have a lot of the rat rain
  LoRA-Tuned :  Have a lot o

In [13]:
# ── Load inference models ──────────────────────────────────────────────────────
print("Loading inference model (LoRA fine-tuned)...")

# Load fresh base model + attach saved LoRA adapter (no patched forward)
inf_base = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    device_map="auto" if DEVICE == "cuda" else None,
)
inf_model = PeftModel.from_pretrained(inf_base, OUTPUT_DIR)
inf_model.eval()
inf_model.config.use_cache = True

print("Loading baseline model for comparison...")
baseline = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    device_map="auto" if DEVICE == "cuda" else None,
)
baseline.eval()
baseline.config.use_cache = True


# ── Transcription function ─────────────────────────────────────────────────────
def transcribe(audio_file):
    if audio_file is None:
        return "⚠️ No file uploaded.", "⚠️ No file uploaded."

    # Load & resample to 16 kHz mono
    path = audio_file if isinstance(audio_file, str) else audio_file.name
    waveform, sr = librosa.load(path, sr=16000, mono=True)

    feats = processor.feature_extractor(
        waveform, sampling_rate=16000
    ).input_features[0]
    feat_tensor = torch.tensor([feats]).to(DEVICE)

    gen_kwargs = dict(
    input_features = feat_tensor,
    language       = "english",
    task           = "transcribe",
    max_new_tokens = 225,
    num_beams      = 3,
)

    with torch.no_grad():
        tuned_ids    = inf_model.generate(**gen_kwargs)
        baseline_ids = baseline.generate(**gen_kwargs)

    tuned_text    = processor.batch_decode(tuned_ids,    skip_special_tokens=True)[0]
    baseline_text = processor.batch_decode(baseline_ids, skip_special_tokens=True)[0]

    return tuned_text, baseline_text


# ── UI styling ─────────────────────────────────────────────────────────────────
CSS = """
/* ─── Global ─────────────────────────────────────────────────────────── */
body, .gradio-container {
    background-color: #09090b !important;
    color: #e4e4e7 !important;
    font-family: 'Inter', system-ui, sans-serif !important;
}

/* ─── Title gradient (teal → violet instead of purple → cyan) ────────── */
.app-title h1 {
    background: linear-gradient(135deg, #2dd4bf 0%, #818cf8 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    font-weight: 900 !important;
    font-size: 2.4rem !important;
    letter-spacing: -0.03em;
}
.app-title h3 {
    color: #71717a !important;
    font-weight: 400 !important;
    margin-top: 4px !important;
}

/* ─── Cards ──────────────────────────────────────────────────────────── */
.card {
    background: #18181b !important;
    border: 1px solid #27272a !important;
    border-radius: 14px !important;
    padding: 22px !important;
    box-shadow: 0 8px 28px rgba(0,0,0,0.5) !important;
}

/* ─── Accent side-borders ────────────────────────────────────────────── */
.tuned-card  { border-left: 4px solid #2dd4bf !important; }  /* teal   */
.base-card   { border-left: 4px solid #6366f1 !important; }  /* indigo */
.info-card   { border-left: 4px solid #f59e0b !important; }  /* amber  */

/* ─── Section headers ────────────────────────────────────────────────── */
.section-label {
    font-size: 0.75rem !important;
    letter-spacing: 0.1em !important;
    text-transform: uppercase !important;
    color: #52525b !important;
    margin-bottom: 8px !important;
}

/* ─── Buttons ────────────────────────────────────────────────────────── */
.run-btn {
    background: linear-gradient(135deg, #2dd4bf, #818cf8) !important;
    color: #09090b !important;
    font-weight: 700 !important;
    border: none !important;
    border-radius: 8px !important;
}
.run-btn:hover { opacity: 0.88 !important; }

.reset-btn {
    background: #27272a !important;
    color: #a1a1aa !important;
    border: 1px solid #3f3f46 !important;
    border-radius: 8px !important;
}

/* ─── Divider ────────────────────────────────────────────────────────── */
.divider { border-color: #27272a !important; margin: 20px 0 !important; }

footer { display: none !important; }
"""

# Force dark theme on load
JS_DARK = """
function init() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
"""

# ── Build Gradio app ───────────────────────────────────────────────────────────
with gr.Blocks(theme=gr.themes.Soft(), css=CSS, js=JS_DARK) as demo:

    # Header
    with gr.Row():
        with gr.Column(scale=5, elem_classes=["app-title"]):
            gr.Markdown(
                """# 🎵 AutoLyrics
### Fine-tuned ASR for Singing — Whisper-Small + LoRA"""
            )

    gr.HTML("<hr class='divider'>")

    # Main row
    with gr.Row(equal_height=False):

        # ── Left: upload + controls ──────────────────────────────────────────
        with gr.Column(scale=1, elem_classes=["card"]):
            gr.Markdown("<p class='section-label'>Input</p>")
            gr.Markdown("### 🎤 Upload Audio")
            audio_in = gr.File(
                label="Music clip or singing audio",
                file_types=[".mp3", ".wav", ".ogg", ".flac", ".m4a", ".mpeg"],
                interactive=True,
            )
            gr.HTML("<div style='height:14px'></div>")
            run_btn   = gr.Button("▶  Transcribe", variant="primary",   elem_classes=["run-btn"])
            reset_btn = gr.Button("↺  Reset",      variant="secondary",  elem_classes=["reset-btn"])

        # ── Right: output panels ──────────────────────────────────────────────
        with gr.Column(scale=2):
            with gr.Row():
                with gr.Column(elem_classes=["card", "tuned-card"]):
                    gr.Markdown("<p class='section-label'>Fine-Tuned Output</p>")
                    gr.Markdown("#### 🟢 AutoLyrics (LoRA Fine-Tuned)")
                    out_tuned = gr.Textbox(
                        label="",
                        placeholder="Fine-tuned transcription will appear here…",
                        lines=3,
                        show_copy_button=True,
                        container=False,
                    )

                with gr.Column(elem_classes=["card", "base-card"]):
                    gr.Markdown("<p class='section-label'>Baseline Output</p>")
                    gr.Markdown("#### 🔵 Whisper-Small (Zero-Shot)")
                    out_base = gr.Textbox(
                        label="",
                        placeholder="Baseline transcription will appear here…",
                        lines=3,
                        show_copy_button=True,
                        container=False,
                    )

    # Info footer
    gr.HTML("<div style='height:14px'></div>")
    with gr.Row(elem_classes=["card", "info-card"]):
        gr.Markdown(
            """
##### ⚙️ Technical Details
| Component | Detail |
|---|---|
| **Base model** | `openai/whisper-small` |
| **Adapter** | LoRA — target: `q_proj`, `v_proj` · rank: 32 · α: 64 |
| **Audio pipeline** | librosa → mono 16 kHz → log-mel spectrogram |
| **Decoding** | Beam search (width = 3), max 225 tokens |
| **Evaluation metrics** | WER · CER · Relative WER reduction (target > 15%) |
"""
        )

    # ── Event wiring ──────────────────────────────────────────────────────────
    run_btn.click(
        fn      = transcribe,
        inputs  = [audio_in],
        outputs = [out_tuned, out_base],
    )
    reset_btn.click(
        fn      = lambda: (None, "", ""),
        inputs  = None,
        outputs = [audio_in, out_tuned, out_base],
    )

demo.launch(share=True)

Loading inference model (LoRA fine-tuned)...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading baseline model for comparison...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

/tmp/ipykernel_58/545774835.py:136: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=CSS, js=JS_DARK) as demo:
/tmp/ipykernel_58/545774835.py:136: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=CSS, js=JS_DARK) as demo:
/tmp/ipykernel_58/545774835.py:136: DeprecationWarning: The 'js' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'js' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=CSS, js=JS_DARK) as demo:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://dddf8f5873b219ef97.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/tmp/ipykernel_58/545774835.py:34: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  feat_tensor = torch.tensor([feats]).to(DEVICE)
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
